# Trainer solutions — do not distribute
Executed cell-by-cell against the shipped historical CSV files and SQLite database.

In [1]:
from pathlib import Path
import csv, sqlite3, json

# Locate the dataset whether the kernel starts in this folder or its parent.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "orders.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Start the kernel inside the extracted Kalpa_Activity1 folder or its notebooks folder.")

def read_table(name, integer_columns=()):
    with (ROOT / "data" / (name + ".csv")).open(newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    for row in rows:
        for field in integer_columns:
            row[field] = int(row[field])
    return rows

orders = read_table("orders", ["gross_amount", "discount_amount", "refund_amount"])
order_costs = read_table("order_costs", ["product_cost", "fulfilment_cost", "payment_cost"])
marketing_spend = read_table("marketing_spend", ["amount"])
periods = read_table("periods")
customers = read_table("customers")
channels = read_table("channels")
db = sqlite3.connect("file:" + str(ROOT / "kalpa.sqlite") + "?mode=ro", uri=True)
db.row_factory = sqlite3.Row

def run_sql(query):
    return [dict(row) for row in db.execute(query)]

print("Data loaded. Monetary values are integer INR. Database connection is read-only.")
print(orders[0])

Data loaded. Monetary values are integer INR. Database connection is read-only.
{'order_id': 'O0000025', 'customer_id': 'C000007', 'period_id': 'FY2024_25', 'order_date': '2024-04-01', 'channel_id': 'STORE', 'status': 'COMPLETED', 'gross_amount': 163890, 'discount_amount': 8560, 'refund_amount': 0}


## Python: both exercises

In [2]:
# Independent calculation from raw CSV records; no target values embedded.
from collections import defaultdict
cost_by_order = {row["order_id"]: row for row in order_costs}
python_results = {}
for period in periods:
    pid = period["period_id"]
    year_orders = [row for row in orders if row["period_id"] == pid]
    completed = [row for row in year_orders if row["status"] == "COMPLETED"]
    buyers = {row["customer_id"] for row in completed}
    revenue = sum(row["gross_amount"] - row["discount_amount"] - row["refund_amount"] for row in completed)
    n_orders = len(completed)
    expenses = {name: sum(cost_by_order[row["order_id"]][name] for row in year_orders)
                for name in ["product_cost", "fulfilment_cost", "payment_cost"]}
    marketing = sum(row["amount"] for row in marketing_spend if row["period_id"] == pid)
    total_cost = sum(expenses.values()) + marketing
    contribution = revenue - total_cost
    margin = contribution / revenue
    python_results[pid] = dict(revenue=revenue, completed_orders=n_orders,
        purchasing_customers=len(buyers), orders_per_customer=n_orders/len(buyers),
        revenue_per_order=revenue/n_orders, **expenses, marketing_cost=marketing,
        total_cost=total_cost, contribution=contribution,
        contribution_margin_pct=margin*100, guardrail="MET" if margin >= .20 else "BREACHED")
print(json.dumps(python_results, indent=2))

previous = python_results["FY2024_25"]
current = python_results["FY2025_26"]
changes = {name: 100 * (current[name]/previous[name] - 1) for name in
           ["revenue", "completed_orders", "purchasing_customers", "orders_per_customer", "revenue_per_order", "total_cost", "contribution"]}
changes["margin_change_percentage_points"] = current["contribution_margin_pct"] - previous["contribution_margin_pct"]
print(json.dumps(changes, indent=2))

{
  "FY2024_25": {
    "revenue": 1000000000,
    "completed_orders": 50000,
    "purchasing_customers": 10000,
    "orders_per_customer": 5.0,
    "revenue_per_order": 20000.0,
    "product_cost": 680000000,
    "fulfilment_cost": 40000000,
    "payment_cost": 10000000,
    "marketing_cost": 70000000,
    "total_cost": 800000000,
    "contribution": 200000000,
    "contribution_margin_pct": 20.0,
    "guardrail": "MET"
  },
  "FY2025_26": {
    "revenue": 1040000000,
    "completed_orders": 52000,
    "purchasing_customers": 13000,
    "orders_per_customer": 4.0,
    "revenue_per_order": 20000.0,
    "product_cost": 716000000,
    "fulfilment_cost": 57200000,
    "payment_cost": 15600000,
    "marketing_cost": 95200000,
    "total_cost": 884000000,
    "contribution": 156000000,
    "contribution_margin_pct": 15.0,
    "guardrail": "BREACHED"
  }
}
{
  "revenue": 4.0000000000000036,
  "completed_orders": 4.0000000000000036,
  "purchasing_customers": 30.000000000000004,
  "orders_per_c

## SQL: Exercise 1

In [3]:
query = '-- Trainer solution. All money is whole INR. The view filters completed orders.\nSELECT p.period_label,\n       COUNT(*) AS completed_orders,\n       COUNT(DISTINCT o.customer_id) AS purchasing_customers,\n       SUM(o.net_revenue) AS revenue,\n       ROUND(1.0 * COUNT(*) / COUNT(DISTINCT o.customer_id), 4) AS orders_per_customer,\n       ROUND(1.0 * SUM(o.net_revenue) / COUNT(*), 2) AS revenue_per_order\nFROM eligible_orders o\nJOIN periods p ON p.period_id = o.period_id\nGROUP BY p.period_id, p.period_label\nORDER BY p.period_id;\n'
print(json.dumps(run_sql(query), indent=2))

[
  {
    "period_label": "FY 2024-25",
    "completed_orders": 50000,
    "purchasing_customers": 10000,
    "revenue": 1000000000,
    "orders_per_customer": 5.0,
    "revenue_per_order": 20000.0
  },
  {
    "period_label": "FY 2025-26",
    "completed_orders": 52000,
    "purchasing_customers": 13000,
    "revenue": 1040000000,
    "orders_per_customer": 4.0,
    "revenue_per_order": 20000.0
  }
]


## SQL: Exercise 2

In [4]:
query = "-- Aggregate each grain separately BEFORE joining. Spend has multiple rows per period.\nWITH revenue AS (\n    SELECT period_id, SUM(net_revenue) AS revenue\n    FROM eligible_orders GROUP BY period_id\n), order_expenses AS (\n    SELECT o.period_id, SUM(c.product_cost) AS product_cost,\n           SUM(c.fulfilment_cost) AS fulfilment_cost,\n           SUM(c.payment_cost) AS payment_cost\n    FROM orders o JOIN order_costs c ON c.order_id = o.order_id\n    GROUP BY o.period_id\n), marketing AS (\n    SELECT period_id, SUM(amount) AS marketing_cost\n    FROM marketing_spend GROUP BY period_id\n)\nSELECT p.period_label, r.revenue, c.product_cost, c.fulfilment_cost,\n       c.payment_cost, m.marketing_cost,\n       c.product_cost+c.fulfilment_cost+c.payment_cost+m.marketing_cost AS total_cost,\n       r.revenue-c.product_cost-c.fulfilment_cost-c.payment_cost-m.marketing_cost AS contribution,\n       ROUND(100.0*(r.revenue-c.product_cost-c.fulfilment_cost-c.payment_cost-m.marketing_cost)/r.revenue,2) AS contribution_margin_pct,\n       CASE WHEN 1.0*(r.revenue-c.product_cost-c.fulfilment_cost-c.payment_cost-m.marketing_cost)/r.revenue >= 0.20 THEN 'MET' ELSE 'BREACHED' END AS margin_floor\nFROM revenue r JOIN order_expenses c ON c.period_id=r.period_id\nJOIN marketing m ON m.period_id=r.period_id\nJOIN periods p ON p.period_id=r.period_id\nORDER BY p.period_id;\n"
print(json.dumps(run_sql(query), indent=2))

[
  {
    "period_label": "FY 2024-25",
    "revenue": 1000000000,
    "product_cost": 680000000,
    "fulfilment_cost": 40000000,
    "payment_cost": 10000000,
    "marketing_cost": 70000000,
    "total_cost": 800000000,
    "contribution": 200000000,
    "contribution_margin_pct": 20.0,
    "margin_floor": "MET"
  },
  {
    "period_label": "FY 2025-26",
    "revenue": 1040000000,
    "product_cost": 716000000,
    "fulfilment_cost": 57200000,
    "payment_cost": 15600000,
    "marketing_cost": 95200000,
    "total_cost": 884000000,
    "contribution": 156000000,
    "contribution_margin_pct": 15.0,
    "margin_floor": "BREACHED"
  }
]


## Interpretation
Revenue +4%, buyers +30%, frequency −20%, average order value unchanged. The average does not describe every buyer, and changing customer mix can affect it. Margin falls 20% to 15%; contribution falls 22%. Costs grew 10.5%. The 20% floor is breached in the current year. These observations justify investigation, not a causal explanation or a claim that future acquisition cannot work.